# Chapter 8.6 - Residual Networks (ResNet) and ResNeXt

ResNet changes the default function a block has to learn. Instead of asking stacked layers to learn a full transformation from scratch, a residual block learns an update added to the input. ResNeXt extends this family with grouped convolutions for a different capacity-compute tradeoff.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs. Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

## You are done when you can

- explain why residual blocks make identity mappings easier
- implement residual addition with and without projection
- trace shape changes through a tiny ResNet
- explain grouped convolution in ResNeXt terms
- debug residual addition when the two paths have incompatible shapes


In [ ]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current


## 8.6.0 The Problem This Notebook Solves

Deeper networks should be more expressive, but they can become harder to optimize. ResNet's core idea is to make a block learn a residual update:

```text
output = input + learned_update(input)
```

If the best behavior is close to "do nothing", the learned update can move toward zero. This gives the architecture an easier path to identity mappings than asking several layers to directly learn identity from scratch.


## 8.6.1 Residual Addition Requires Matching Shapes

Addition is stricter than concatenation. To compute `Y + X`, both tensors must have the same shape or be broadcast-compatible. In ResNet blocks, the intended case is same shape:

```text
main path output shape == shortcut path output shape
```

The block below keeps channel count and spatial size unchanged.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_projection=False):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        if use_projection:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
        else:
            self.shortcut = nn.Identity()

    def forward(self, X):
        return F.relu(self.main(X) + self.shortcut(X))


block = ResidualBlock(8, 8)
X = torch.randn(2, 8, 16, 16)
Y = block(X)

print("output:", shape(Y))
assert shape(Y) == shape(X)


## 8.6.2 Projection Shortcuts Change Shape Deliberately

When a residual stage changes channel count or spatial size, the shortcut path must change shape too. A 1 by 1 convolution can project the input into the required shape.

Here the main path uses stride 2 and changes channels from 8 to 16. The shortcut must do the same.


In [ ]:
projection_block = ResidualBlock(8, 16, stride=2, use_projection=True)
X = torch.randn(2, 8, 16, 16)
Y = projection_block(X)

shortcut_Y = projection_block.shortcut(X)
main_Y = projection_block.main(X)

print("main path:", shape(main_Y))
print("shortcut:", shape(shortcut_Y))
print("block output:", shape(Y))

assert shape(main_Y) == shape(shortcut_Y)
assert shape(Y) == (2, 16, 8, 8)


## 8.6.3 Identity Is Easy When the Residual Update Is Zero

This tiny block strips away convolution and batch normalization to isolate the function-class idea. If the learned update is zero, the residual block returns the input, up to the final activation.

For nonnegative inputs, `relu(X + 0)` equals `X`.


In [ ]:
class ZeroUpdateResidual(nn.Module):
    def forward(self, X):
        update = torch.zeros_like(X)
        return F.relu(X + update)


X = torch.rand(2, 3, 4, 4)
Y = ZeroUpdateResidual()(X)

print("max difference:", (Y - X).abs().max().item())
assert torch.equal(Y, X)


## 8.6.4 A Tiny ResNet Uses Residual Blocks as Stages

A ResNet is not just one residual block. It uses stages:

```text
stem -> residual stage -> residual stage -> global average pool -> classifier
```

The first block of a new stage often downsamples and changes channel count with a projection shortcut. Later blocks in the same stage keep shape.


In [ ]:
tiny_resnet = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.BatchNorm2d(8), nn.ReLU(),
    ResidualBlock(8, 8),
    ResidualBlock(8, 16, stride=2, use_projection=True),
    ResidualBlock(16, 16),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(16, 10),
)

X = torch.randn(2, 1, 32, 32)
rows, logits = trace_module_shapes(tiny_resnet, X)
for row in rows:
    print(row)

assert shape(logits) == (2, 10)


## 8.6.5 ResNeXt Uses Grouped Convolutions

Grouped convolution splits input channels into groups. Each group is convolved separately, and the results are concatenated as output channels.

With `groups=4`, a convolution with 16 input channels behaves like four smaller convolutions over 4 channels each. This can reduce parameters and computation while preserving multiple parallel transformation paths. ResNeXt uses this idea inside residual-style blocks.


In [ ]:
dense_conv = nn.Conv2d(16, 32, kernel_size=3, padding=1, groups=1, bias=False)
grouped_conv = nn.Conv2d(16, 32, kernel_size=3, padding=1, groups=4, bias=False)

X = torch.randn(2, 16, 8, 8)
Y = grouped_conv(X)

print("dense weight shape:", shape(dense_conv.weight))
print("grouped weight shape:", shape(grouped_conv.weight))
print("dense params:", count_parameters(dense_conv))
print("grouped params:", count_parameters(grouped_conv))

assert shape(Y) == (2, 32, 8, 8)
assert count_parameters(grouped_conv) < count_parameters(dense_conv)


## 8.6.6 Break It Deliberately: Add Tensors With Different Channels

If the main path changes channel count and the shortcut remains identity, residual addition fails. The right fix is not to silence the error; the right fix is to add a projection shortcut or keep the shapes unchanged.


In [ ]:
bad_block = ResidualBlock(8, 16, use_projection=False)

try:
    bad_block(torch.randn(2, 8, 16, 16))
except RuntimeError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected residual addition to fail with channel mismatch")


## 8.6 Checkpoint

Answer these before moving on. Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. What function does a residual block learn?
2. Why does residual addition require a strict shape contract?
3. When do we need a projection shortcut?
4. Why can residual connections make identity mappings easier to represent?
5. What does grouped convolution change compared with a dense convolution?
